In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import requests
from pyspark.sql.functions import *
from pyspark.sql.types import *
from notebookutils import mssparkutils



StatementMeta(, a6c12cac-d704-44e1-a0f6-3f5738a288d8, 4, Finished, Available, Finished, False)

In [3]:
# Telechargement de ipsum
url="https://raw.githubusercontent.com/stamparm/ipsum/master/ipsum.txt"
response = requests.get(url)
response.raise_for_status()

# Parser avec validation stricte
lines = []
for line in response.text.splitlines():
    line = line.strip()
    if not line:
        continue
    parts = line.split()
    if len(parts) >= 2 and parts[0].startswith(tuple('0123456789')):
        try:
            ip = parts[0]
            score = int(parts[1])
            lines.append((ip, score))
        except ValueError:
            continue

print(f"Parsed {len(lines)} IPs from ipsum.txt")

df_ips = spark.createDataFrame(lines, ["ip_address", "blacklist_hits"])

df_ips = df_ips \
    .withColumn("threat_level", 
        when(col("blacklist_hits") >= 8, "critical")
        .when(col("blacklist_hits") >= 6, "high")
        .when(col("blacklist_hits") >= 4, "medium")
        .otherwise("low")
    ) \
    .withColumn("source_date", current_timestamp()) \
    .withColumn("update_date", current_date())

df_ips.write.format("delta").mode("overwrite").saveAsTable("bronze_ipsum")

StatementMeta(, a6c12cac-d704-44e1-a0f6-3f5738a288d8, 5, Finished, Available, Finished, False)

Parsed 122381 IPs from ipsum.txt


In [4]:
df_ips.show(20)


StatementMeta(, a6c12cac-d704-44e1-a0f6-3f5738a288d8, 6, Finished, Available, Finished, False)

+--------------+--------------+------------+--------------------+-----------+
|    ip_address|blacklist_hits|threat_level|         source_date|update_date|
+--------------+--------------+------------+--------------------+-----------+
|    2.27.53.96|             9|    critical|2026-04-05 21:44:...| 2026-04-05|
| 23.94.213.157|             9|    critical|2026-04-05 21:44:...| 2026-04-05|
|    45.91.64.7|             9|    critical|2026-04-05 21:44:...| 2026-04-05|
|  80.82.77.139|             9|    critical|2026-04-05 21:44:...| 2026-04-05|
| 93.174.95.106|             9|    critical|2026-04-05 21:44:...| 2026-04-05|
| 182.23.36.166|             9|    critical|2026-04-05 21:44:...| 2026-04-05|
|66.240.192.138|             8|    critical|2026-04-05 21:44:...| 2026-04-05|
|  71.6.135.131|             8|    critical|2026-04-05 21:44:...| 2026-04-05|
|   80.82.77.33|             8|    critical|2026-04-05 21:44:...| 2026-04-05|
|118.70.178.158|             8|    critical|2026-04-05 21:44:...

In [5]:
# Sources
logs = spark.table("Bronze_security_logs")
threat = spark.table("bronze_ipsum").hint("broadcast")

# Jointure + enrichissement
silver_enriched = (
    logs
    .join(
        threat,
        (logs.src_ip == threat.ip_address) | 
        (logs.dst_ip == threat.ip_address),
        "left"
    )
    .withColumn("matched_ip", threat.ip_address)
    .withColumn("threat_score", threat.blacklist_hits)
    .withColumn("threat_score_clean", coalesce(threat.blacklist_hits, lit(0)))
    .withColumn("is_malicious", col("threat_score_clean") > 0)
    .withColumn(
        "threat_level",
        when(col("threat_score_clean") >= 8, "CRITICAL")
        .when(col("threat_score_clean") >= 6, "HIGH")
        .when(col("threat_score_clean") >= 4, "MEDIUM")
        .otherwise("LOW")
    )
    .withColumn("enriched_at", current_timestamp())
    # Sélectionne TOUTES les colonnes (logs + enrichissement)
    .select(
        [col(c) for c in logs.columns] + 
        ["matched_ip", "threat_score", "threat_score_clean", "is_malicious", "threat_level", "enriched_at"]
    )
)

# Sauvegarde
silver_enriched.write.format("delta").mode("overwrite").saveAsTable("silver_enriched_logs")

StatementMeta(, a6c12cac-d704-44e1-a0f6-3f5738a288d8, 7, Finished, Available, Finished, False)

In [6]:
silver_enriched.show(10)


StatementMeta(, a6c12cac-d704-44e1-a0f6-3f5738a288d8, 8, Finished, Available, Finished, False)

+--------------------+------------+-----------------+--------+---------------+--------+---------------+--------+--------+------------+--------------------+--------------------+---------------------+-----------+--------------------+----------+------------+------------------+------------+------------+--------------------+
|          event_time| logger_name|         severity|protocol|         src_ip|src_port|         dst_ip|dst_port|log_type|parse_status|         raw_message|         ingest_time|EventProcessedUtcTime|PartitionId|EventEnqueuedUtcTime|matched_ip|threat_score|threat_score_clean|is_malicious|threat_level|         enriched_at|
+--------------------+------------+-----------------+--------+---------------+--------+---------------+--------+--------+------------+--------------------+--------------------+---------------------+-----------+--------------------+----------+------------+------------------+------------+------------+--------------------+
|2026-04-05 20:12:...|ids_logger_1

In [10]:
# Gold : analyse horaire
from pyspark.sql.functions import sum as spark_sum
gold_analysis = (
    spark.table("silver_enriched_logs")
    .withColumn("event_hour", date_trunc("hour", col("ingest_time")))
    .groupBy(
        "event_hour",
        "protocol",
        "severity"
    )
    .agg(
        count("*").alias("total_events"),
        spark_sum(when(col("is_malicious"), 1).otherwise(0)).alias("malicious_events"),
        avg("threat_score_clean").alias("avg_threat_score")
    )
    .withColumn("malicious_ratio", col("malicious_events") / col("total_events"))
    .orderBy(desc("event_hour"), desc("malicious_ratio"))
)

gold_analysis.write.format("delta").mode("overwrite").saveAsTable("gold_threat_analysis")

StatementMeta(, a6c12cac-d704-44e1-a0f6-3f5738a288d8, 12, Finished, Available, Finished, False)